In [1]:
from pathlib import Path
import copy
import json
import sys
import numpy as np

ROOT = Path('/home/baiyu/LearnStageConstraints')
PLANNER_SRC = ROOT / 'robot/stage_cons_iiwa14/ros_ws/src/stage_constraint_planner/src'
if str(PLANNER_SRC) not in sys.path:
    sys.path.insert(0, str(PLANNER_SRC))
from stage_constraint_planner.constraint_artifact import configure_planning_profile

run_names = ['Bssf3', 'Bsss1', 'Bsss2', 'Bsss4']
runs_root = ROOT / 'robot/final_video_runs/BarClean'
artifact_path = ROOT / 'outputs/map_balanced_pooled/BarClean/method_seed_000/learned_constraints.json'
config_path = ROOT / 'robot/stage_cons_iiwa14/ros_ws/src/stage_constraint_planner/config/bar_clean_true.json'
config = json.loads(config_path.read_text())
configure_planning_profile(config, 'BarClean', artifact_path, ROOT / 'outputs')
planning_terms = config['constraint_terms']
s4_terms = [term for term in planning_terms if int(term['stage']) == 3]
print('python', sys.executable)
print('runs', run_names)
print('S4 deployed terms', [{key: term[key] for key in ('feature_name', 'semantics', 'value', 'scale', 'weight')} for term in s4_terms])

python /home/baiyu/miniforge3/envs/segment/bin/python
runs ['Bssf3', 'Bsss1', 'Bsss2', 'Bsss4']
S4 deployed terms [{'feature_name': 'obs_dist', 'semantics': 'target_value', 'value': 0.3972037160599975, 'scale': 0.0234, 'weight': 6.0}, {'feature_name': 'table_dist', 'semantics': 'target_value', 'value': 0.0662575107363458, 'scale': 0.003, 'weight': 1.5}, {'feature_name': 'axial_offset', 'semantics': 'target_value', 'value': -0.031846260346327177, 'scale': 0.003, 'weight': 1.5}, {'feature_name': 'tool_pitch', 'semantics': 'target_value', 'value': 1.5873664972040709, 'scale': 0.0482, 'weight': 1.0}, {'feature_name': 'tool_roll', 'semantics': 'target_value', 'value': -0.04001447648337279, 'scale': 0.0411, 'weight': 1.0}, {'feature_name': 'tool_yaw', 'semantics': 'target_value', 'value': -0.6999839052468153, 'scale': 0.035, 'weight': 2.0}]


In [2]:
def wrap_angle(values):
    return (np.asarray(values) + np.pi) % (2.0 * np.pi) - np.pi

def audit_series(vis, source_key, run_name):
    series = vis[source_key]
    names = [item['name'] for item in series['schema']]
    samples = np.asarray(series['samples'], dtype=float)
    times = samples[:, 0]
    values = samples[:, 1:]
    core_start = float(vis['stage_transition_end_times'][2])
    core_end = float(vis['stage_boundary_times'][3])
    mask = (times >= core_start) & (times <= core_end)
    rows = []
    for term in s4_terms:
        name = term['feature_name']
        observed = values[mask, names.index(name)]
        error = observed - float(term['value'])
        if name == 'tool_yaw':
            error = wrap_angle(error)
        rows.append({
            'run': run_name,
            'source': 'planned' if source_key == 'planned_feature_series' else 'executed',
            'feature': name,
            'n': int(mask.sum()),
            'mean': float(np.mean(observed)),
            'target': float(term['value']),
            'mae': float(np.mean(np.abs(error))),
            'rmse': float(np.sqrt(np.mean(error ** 2))),
            'max_abs': float(np.max(np.abs(error))),
            'rmse_over_scale': float(np.sqrt(np.mean(error ** 2)) / float(term['scale'])),
            'within_1scale': float(np.mean(np.abs(error) <= float(term['scale']))),
        })
    return rows

rows = []
run_facts = []
for run_name in run_names:
    run_dir = runs_root / run_name
    vis = json.loads((run_dir / 'visualization.json').read_text())
    metadata = json.loads((run_dir / 'metadata.json').read_text())
    run_facts.append({'run': run_name, 'constraint_source': metadata['constraint_source'], 'execution_source': vis['feature_series']['source'], 'planned_source': vis['planned_feature_series']['source'], 'execution_n': len(vis['feature_series']['samples']), 'planned_n': len(vis['planned_feature_series']['samples']), 's4_core_s': [vis['stage_transition_end_times'][2], vis['stage_boundary_times'][3]]})
    rows.extend(audit_series(vis, 'planned_feature_series', run_name))
    rows.extend(audit_series(vis, 'feature_series', run_name))

print('RUN FACTS')
for fact in run_facts:
    print(fact)
print()
print('S4 CORE AUDIT: run source feature mean target MAE RMSE RMSE/scale within1scale')
for row in rows:
    print(f"{row['run']:5s} {row['source']:8s} {row['feature']:15s} {row['mean']:+.4f} {row['target']:+.4f} {row['mae']:.4f} {row['rmse']:.4f} {row['rmse_over_scale']:.2f} {row['within_1scale']:.2f}")

RUN FACTS
{'run': 'Bssf3', 'constraint_source': '/learned_constraints/map_balanced_pooled/BarClean/method_seed_000/learned_constraints.json', 'execution_source': 'real_tf/BarClean', 'planned_source': 'stage_constraint_planner/BarClean', 'execution_n': 202, 'planned_n': 301, 's4_core_s': [11.098245637854983, 12.853248572656913]}
{'run': 'Bsss1', 'constraint_source': '/learned_constraints/map_balanced_pooled/BarClean/method_seed_000/learned_constraints.json', 'execution_source': 'real_tf/BarClean', 'planned_source': 'stage_constraint_planner/BarClean', 'execution_n': 181, 'planned_n': 253, 's4_core_s': [8.508163414836908, 10.263349530414388]}
{'run': 'Bsss2', 'constraint_source': '/learned_constraints/map_balanced_pooled/BarClean/method_seed_000/learned_constraints.json', 'execution_source': 'real_tf/BarClean', 'planned_source': 'stage_constraint_planner/BarClean', 'execution_n': 205, 'planned_n': 297, 's4_core_s': [10.823468467255802, 12.578733757457883]}
{'run': 'Bsss4', 'constraint_so

In [3]:
print('S4 OBS GEOMETRY BY RUN')
for run_name in run_names:
    vis = json.loads((runs_root / run_name / 'visualization.json').read_text())
    series = vis['planned_feature_series']
    names = [item['name'] for item in series['schema']]
    samples = np.asarray(series['samples'], dtype=float)
    obs = samples[:, 1 + names.index('obs_dist')]
    times = samples[:, 0]
    core_start = float(vis['stage_transition_end_times'][2])
    core_end = float(vis['stage_boundary_times'][3])
    core = obs[(times >= core_start) & (times <= core_end)]
    endpoint_index = int(vis['stage_boundary_indices'][3])
    endpoint_value = float(obs[endpoint_index])
    print({'run': run_name, 'target': s4_terms[0]['value'], 'core_min': float(core.min()), 'core_max': float(core.max()), 'endpoint_index': endpoint_index, 'endpoint_obs_dist': endpoint_value, 'endpoint_abs_error': abs(endpoint_value - float(s4_terms[0]['value']))})

print()
print('AGGREGATED RMSE OVER FOUR RUNS')
for source in ('planned', 'executed'):
    for term in s4_terms:
        values = [row['rmse'] for row in rows if row['source'] == source and row['feature'] == term['feature_name']]
        normalized = [row['rmse_over_scale'] for row in rows if row['source'] == source and row['feature'] == term['feature_name']]
        print(f"{source:8s} {term['feature_name']:15s} rmse_mean={np.mean(values):.4f} normalized_mean={np.mean(normalized):.2f}")

S4 OBS GEOMETRY BY RUN
{'run': 'Bssf3', 'target': 0.3972037160599975, 'core_min': 0.42310317239833295, 'core_max': 0.45195816533084615, 'endpoint_index': 222, 'endpoint_obs_dist': 0.45195816533084615, 'endpoint_abs_error': 0.05475444927084866}
{'run': 'Bsss1', 'target': 0.3972037160599975, 'core_min': 0.42310517540616627, 'core_max': 0.45195816533084615, 'endpoint_index': 180, 'endpoint_obs_dist': 0.45195816533084615, 'endpoint_abs_error': 0.05475444927084866}
{'run': 'Bsss2', 'target': 0.3972037160599975, 'core_min': 0.42311442407488037, 'core_max': 0.45195816533084615, 'endpoint_index': 218, 'endpoint_obs_dist': 0.45195816533084615, 'endpoint_abs_error': 0.05475444927084866}
{'run': 'Bsss4', 'target': 0.3972037160599975, 'core_min': 0.4231271177259554, 'core_max': 0.45195816533084615, 'endpoint_index': 223, 'endpoint_obs_dist': 0.45195816533084615, 'endpoint_abs_error': 0.05475444927084866}

AGGREGATED RMSE OVER FOUR RUNS
planned  obs_dist        rmse_mean=0.0377 normalized_mean=1.61

In [4]:
import math
import time
from stage_constraint_planner.optimizer import StageConstraintTrajectoryOptimizer

def make_optimizer(task_config):
    planner = task_config['planner']
    return StageConstraintTrajectoryOptimizer(
        task_config,
        control_spacing=float(planner['control_spacing_m']),
        output_spacing=float(planner['output_spacing_m']),
        output_axis_spacing=math.radians(float(planner['output_axis_spacing_deg'])),
        min_control_points=int(planner['min_control_points']),
        max_control_points=int(planner['max_control_points']),
        max_nfev=int(planner['max_nfev']),
        multi_start=int(planner['multi_start']),
    )

def pose_from_result(value):
    return np.asarray([value['x'], value['y'], value['z'], value['qx'], value['qy'], value['qz'], value['qw']], dtype=float)

def reconstruct_scene(vis, task_config):
    bar_info = vis['scene_geometry']['bar']
    obs_info = vis['scene_geometry']['obstacle']
    axis = np.asarray(bar_info['axis'], dtype=float)
    yaw = math.atan2(axis[1], axis[0])
    table_z = float(task_config['table_surface_point'][2])
    bar = np.asarray([bar_info['pivot'][0], bar_info['pivot'][1], table_z, 0.0, 0.0, math.sin(yaw / 2.0), math.cos(yaw / 2.0)], dtype=float)
    obstacle = {'type': 'circle', 'center': np.asarray([obs_info['center'][0], obs_info['center'][1], table_z], dtype=float), 'radius': float(obs_info['radius'])}
    return bar, obstacle

def resample_curve(points, count=1001):
    points = np.asarray(points, dtype=float)
    lengths = np.linalg.norm(np.diff(points, axis=0), axis=1)
    cumulative = np.concatenate(([0.0], np.cumsum(lengths)))
    if cumulative[-1] <= 1e-12:
        return np.repeat(points[:1], count, axis=0)
    keep = np.concatenate(([True], np.diff(cumulative) > 1e-12))
    cumulative = cumulative[keep] / cumulative[-1]
    points = points[keep]
    query = np.linspace(0.0, 1.0, count)
    return np.column_stack([np.interp(query, cumulative, points[:, dim]) for dim in range(points.shape[1])])

eq_config = copy.deepcopy(config)
inactive_config = copy.deepcopy(config)
inactive_config['constraint_terms'] = [term for term in inactive_config['constraint_terms'] if not (int(term['stage']) == 3 and str(term['feature_name']) == 'obs_dist')]
eq_optimizer = make_optimizer(eq_config)
inactive_optimizer = make_optimizer(inactive_config)
print('term counts', len(eq_config['constraint_terms']), len(inactive_config['constraint_terms']))

term counts 12 11


In [5]:
validation_name = 'Bssf3'
validation_dir = runs_root / validation_name
validation_result = json.loads((validation_dir / 'result.json').read_text())
validation_vis = json.loads((validation_dir / 'visualization.json').read_text())
validation_start = pose_from_result(validation_result['task']['start'])
validation_goal = pose_from_result(validation_result['task']['goal'])
validation_bar, validation_obstacle = reconstruct_scene(validation_vis, eq_config)
started = time.perf_counter()
validation_eq = eq_optimizer.plan(validation_start, validation_goal, validation_bar, validation_obstacle, bar_lateral_centerline={'type': 'straight'}, seed=2026)
recorded_xy = np.asarray(validation_vis['planned_trace'], dtype=float)
recomputed_xy = np.asarray(validation_eq['positions'], dtype=float)[:, :2]
delta = np.linalg.norm(resample_curve(recorded_xy) - resample_curve(recomputed_xy), axis=1)
print({'elapsed_s': time.perf_counter() - started, 'recorded_n': len(recorded_xy), 'recomputed_n': len(recomputed_xy), 'xy_mean_delta_m': float(delta.mean()), 'xy_max_delta_m': float(delta.max()), 'recomputed_objective': validation_eq['objective'], 'solver_success': validation_eq['solver_success']})

{'elapsed_s': 6.295308572705835, 'recorded_n': 301, 'recomputed_n': 292, 'xy_mean_delta_m': 0.014608955320605367, 'xy_max_delta_m': 0.03660793032029331, 'recomputed_objective': 246.58393448405363, 'solver_success': True}


In [6]:
available_bag_readers = {}
for module_name in ('rosbag', 'rosbags'):
    try:
        module = __import__(module_name)
        available_bag_readers[module_name] = getattr(module, '__file__', 'available')
    except Exception as exc:
        available_bag_readers[module_name] = f'{type(exc).__name__}: {exc}'
print(available_bag_readers)

{'rosbag': "ModuleNotFoundError: No module named 'rosbag'", 'rosbags': None}


In [7]:
from rosbags.highlevel import AnyReader
bag_path = runs_root / 'Bssf3' / 'real_task.bag'
with AnyReader([bag_path]) as reader:
    bag_topics = [(connection.topic, connection.msgtype, connection.msgcount) for connection in reader.connections]
for item in bag_topics:
    print(item)

('/iiwa14/commanding_status', 'std_msgs/msg/Bool', 381)
('/iiwa14/fri_command_mode', 'std_msgs/msg/Int32', 380)
('/iiwa14/real_executor/status', 'std_msgs/msg/String', 81)
('/iiwa14/additional_outputs', 'iiwa_driver/msg/AdditionalOutputs', 4001)
('/stage_cons/plan', 'nav_msgs/msg/Path', 1)
('/tf_static', 'tf2_msgs/msg/TFMessage', 1)
('/stage_cons/plan_orientation_constraints', 'std_msgs/msg/String', 1)
('/vrpn_client_node/baiyu_obs_bar/pose_from_iiwa14', 'geometry_msgs/msg/PoseStamped', 401)
('/iiwa14/PositionTrajectoryController/state', 'control_msgs/msg/JointTrajectoryControllerState', 1002)
('/iiwa14/joint_states', 'sensor_msgs/msg/JointState', 4002)
('/vrpn_client_node/baiyu_bar/pose_from_iiwa14', 'geometry_msgs/msg/PoseStamped', 402)
('/stage_cons/plan_stage_boundaries', 'std_msgs/msg/Int32MultiArray', 1)
('/stage_cons/planner/tracking_status', 'std_msgs/msg/String', 201)
('/tf', 'tf2_msgs/msg/TFMessage', 881)
('/iiwa14/fri_diagnostics', 'iiwa_driver/msg/FriDiagnostics', 20)


In [8]:
from stage_constraint_planner.optimizer import transform_pose

def msg_pose_array(msg):
    pose = msg.pose
    return np.asarray([pose.position.x, pose.position.y, pose.position.z, pose.orientation.x, pose.orientation.y, pose.orientation.z, pose.orientation.w], dtype=float)

scene_rotation = np.asarray([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=float)
scene_translation = np.zeros(3, dtype=float)
first_messages = {}
with AnyReader([bag_path]) as reader:
    wanted = {'/vrpn_client_node/baiyu_bar/pose_from_iiwa14', '/vrpn_client_node/baiyu_obs_bar/pose_from_iiwa14', '/stage_cons/plan'}
    connections = [connection for connection in reader.connections if connection.topic in wanted]
    for connection, timestamp, rawdata in reader.messages(connections=connections):
        if connection.topic in first_messages:
            continue
        first_messages[connection.topic] = (timestamp, reader.deserialize(rawdata, connection.msgtype))

raw_bar = msg_pose_array(first_messages['/vrpn_client_node/baiyu_bar/pose_from_iiwa14'][1])
raw_obstacle = msg_pose_array(first_messages['/vrpn_client_node/baiyu_obs_bar/pose_from_iiwa14'][1])
bag_bar = transform_pose(raw_bar, scene_rotation, scene_translation)
bag_obstacle_pose = transform_pose(raw_obstacle, scene_rotation, scene_translation)
bag_obstacle = {'type': 'circle', 'center': bag_obstacle_pose[:3], 'radius': float(eq_config['obstacle_radius'])}
plan_message = first_messages['/stage_cons/plan'][1]
bag_plan_positions = np.asarray([[pose.pose.position.x, pose.pose.position.y, pose.pose.position.z] for pose in plan_message.poses], dtype=float)
print({'raw_bar': raw_bar.tolist(), 'bag_bar_robot': bag_bar.tolist(), 'bag_obstacle_robot': bag_obstacle_pose.tolist(), 'bag_plan_n': len(bag_plan_positions), 'bag_plan_first': bag_plan_positions[0].tolist(), 'bag_plan_last': bag_plan_positions[-1].tolist()})

{'raw_bar': [-0.05583778, 0.12537, 0.58766645, 0.09234153467712089, 0.7010513825487217, 0.7010513825487217, -0.09234153467712089], 'bag_bar_robot': [0.58766645, -0.05583778, 0.12537, 0.0, 0.0, -0.6087098478716008, 0.7933929172258426], 'bag_obstacle_robot': [0.585671, 0.208104, 0.13437, 0.0, 0.0, 0.0, 1.0], 'bag_plan_n': 301, 'bag_plan_first': [0.5362, 0.4873, 0.33], 'bag_plan_last': [0.5992, -0.3486, 0.33]}


In [9]:
latched = {}
with AnyReader([bag_path]) as reader:
    wanted = {'/stage_cons/plan_stage_boundaries', '/stage_cons/plan_orientation_constraints'}
    connections = [connection for connection in reader.connections if connection.topic in wanted]
    for connection, timestamp, rawdata in reader.messages(connections=connections):
        if connection.topic not in latched:
            latched[connection.topic] = reader.deserialize(rawdata, connection.msgtype)
bag_boundaries = np.asarray(latched['/stage_cons/plan_stage_boundaries'].data, dtype=int)
orientation_payload = json.loads(latched['/stage_cons/plan_orientation_constraints'].data)
task_frame = validation_eq['task_frame']
rotation_task = np.asarray(task_frame['rotation_world_from_task'], dtype=float)
origin_task = np.asarray(task_frame['origin'], dtype=float)
recorded_endpoint_task = (bag_plan_positions[bag_boundaries[:4]] - origin_task) @ rotation_task
print({'boundaries': bag_boundaries.tolist(), 'recorded_endpoint_task': recorded_endpoint_task.tolist(), 'current_artifact_endpoint_positions': np.asarray(eq_config['stage_endpoint_positions_bar']).tolist(), 'orientation_payload_keys': list(orientation_payload)})

{'boundaries': [93, 159, 187, 222, 300], 'recorded_endpoint_task': [[-0.12446279449535434, 0.006533585311021173, 0.09085575011157979], [0.18757579339632277, 0.008214594150773285, 0.09327618241946892], [0.18805567588973265, 0.07654411998632428, 0.07502988507245892], [0.196038870103033, -0.08733889716275552, 0.06627548924976759]], 'current_artifact_endpoint_positions': [[-0.1244627944953543, 0.006533585311021166, 0.12499766198161479], [0.18757579339632277, 0.008214594150773179, 0.09170150854477627], [0.18805567588973263, 0.07654411998632417, 0.07511824405366223], [0.19603887010303298, -0.08733889716275554, 0.07809831128644533]], 'orientation_payload_keys': ['schema_version', 'stamp_ns', 'task_id', 'point_count', 'tool_yaw_active', 'stage_timing', 'approach_obstacle']}


In [10]:
def load_actual_inputs(run_name):
    run_dir = runs_root / run_name
    result = json.loads((run_dir / 'result.json').read_text())
    start = pose_from_result(result['task']['start'])
    goal = pose_from_result(result['task']['goal'])
    poses = {}
    with AnyReader([run_dir / 'real_task.bag']) as reader:
        wanted = {'/vrpn_client_node/baiyu_bar/pose_from_iiwa14', '/vrpn_client_node/baiyu_obs_bar/pose_from_iiwa14'}
        connections = [connection for connection in reader.connections if connection.topic in wanted]
        for connection, timestamp, rawdata in reader.messages(connections=connections):
            if connection.topic not in poses:
                poses[connection.topic] = msg_pose_array(reader.deserialize(rawdata, connection.msgtype))
            if len(poses) == 2:
                break
    bar = transform_pose(poses['/vrpn_client_node/baiyu_bar/pose_from_iiwa14'], scene_rotation, scene_translation)
    obstacle_pose = transform_pose(poses['/vrpn_client_node/baiyu_obs_bar/pose_from_iiwa14'], scene_rotation, scene_translation)
    obstacle = {'type': 'circle', 'center': obstacle_pose[:3], 'radius': float(eq_config['obstacle_radius'])}
    return start, goal, bar, obstacle

def path_length(points):
    return float(np.sum(np.linalg.norm(np.diff(np.asarray(points), axis=0), axis=1)))

def compare_plans(run_name, eq_plan, inactive_plan):
    eq_all = resample_curve(eq_plan['positions'])
    inactive_all = resample_curve(inactive_plan['positions'])
    all_delta = np.linalg.norm(eq_all - inactive_all, axis=1)
    eq_s4 = resample_curve(eq_plan['positions'][np.asarray(eq_plan['stage_labels']) == 3])
    inactive_s4 = resample_curve(inactive_plan['positions'][np.asarray(inactive_plan['stage_labels']) == 3])
    s4_delta = np.linalg.norm(eq_s4 - inactive_s4, axis=1)
    eq_core = np.asarray(eq_plan['stage_constraint_weights'])[:, 3] >= 1.0 - 1e-9
    inactive_core = np.asarray(inactive_plan['stage_constraint_weights'])[:, 3] >= 1.0 - 1e-9
    return {
        'run': run_name,
        'eq_n': len(eq_plan['positions']),
        'inactive_n': len(inactive_plan['positions']),
        'whole_mean_delta_mm': float(1000.0 * all_delta.mean()),
        'whole_max_delta_mm': float(1000.0 * all_delta.max()),
        's4_mean_delta_mm': float(1000.0 * s4_delta.mean()),
        's4_max_delta_mm': float(1000.0 * s4_delta.max()),
        'path_length_eq_m': path_length(eq_plan['positions']),
        'path_length_inactive_m': path_length(inactive_plan['positions']),
        's4_endpoint_delta_mm': float(1000.0 * np.linalg.norm(np.asarray(eq_plan['stage_endpoints_world'])[3] - np.asarray(inactive_plan['stage_endpoints_world'])[3])),
        's4_obs_eq_mean_m': float(np.mean(np.asarray(eq_plan['features']['obs_dist'])[eq_core])),
        's4_obs_inactive_mean_m': float(np.mean(np.asarray(inactive_plan['features']['obs_dist'])[inactive_core])),
        'eq_objective': float(eq_plan['objective']),
        'inactive_objective': float(inactive_plan['objective']),
        'eq_solver_success': bool(eq_plan['solver_success']),
        'inactive_solver_success': bool(inactive_plan['solver_success']),
    }

ab_plans = {}
ab_rows = []
for run_name in run_names:
    start, goal, bar, obstacle = load_actual_inputs(run_name)
    started = time.perf_counter()
    eq_plan = eq_optimizer.plan(start, goal, bar, obstacle, bar_lateral_centerline={'type': 'straight'}, seed=2026)
    inactive_plan = inactive_optimizer.plan(start, goal, bar, obstacle, bar_lateral_centerline={'type': 'straight'}, seed=2026)
    ab_plans[run_name] = {'eq': eq_plan, 'inactive': inactive_plan}
    row = compare_plans(run_name, eq_plan, inactive_plan)
    row['elapsed_s'] = float(time.perf_counter() - started)
    ab_rows.append(row)
    print(row)

{'run': 'Bssf3', 'eq_n': 292, 'inactive_n': 296, 'whole_mean_delta_mm': 8.773940055017992, 'whole_max_delta_mm': 18.83794182184473, 's4_mean_delta_mm': 1.5517954220144878, 's4_max_delta_mm': 2.522182865972113, 'path_length_eq_m': 1.2322755685984328, 'path_length_inactive_m': 1.2523762274908723, 's4_endpoint_delta_mm': 0.0010924656081157114, 's4_obs_eq_mean_m': 0.4324694078136501, 's4_obs_inactive_mean_m': 0.4338115246450192, 'eq_objective': 246.58393448418897, 'inactive_objective': 9.514378953916161, 'eq_solver_success': True, 'inactive_solver_success': True, 'elapsed_s': 28.135057956911623}
{'run': 'Bsss1', 'eq_n': 247, 'inactive_n': 251, 'whole_mean_delta_mm': 8.630710137130977, 'whole_max_delta_mm': 18.070409519577293, 's4_mean_delta_mm': 1.555725544549316, 's4_max_delta_mm': 2.532282140530071, 'path_length_eq_m': 0.9990040058091365, 'path_length_inactive_m': 1.0196568251727207, 's4_endpoint_delta_mm': 0.0007552800237453638, 's4_obs_eq_mean_m': 0.43246820514513207, 's4_obs_inactive_

In [11]:
def symmetric_hausdorff(first, second):
    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)
    distances = np.linalg.norm(first[:, None, :] - second[None, :, :], axis=2)
    return float(max(np.max(np.min(distances, axis=1)), np.max(np.min(distances, axis=0))))

stage_rows = []
for run_name in run_names:
    eq_plan = ab_plans[run_name]['eq']
    inactive_plan = ab_plans[run_name]['inactive']
    for stage in range(5):
        eq_points = np.asarray(eq_plan['positions'])[np.asarray(eq_plan['stage_labels']) == stage]
        inactive_points = np.asarray(inactive_plan['positions'])[np.asarray(inactive_plan['stage_labels']) == stage]
        aligned_delta = np.linalg.norm(resample_curve(eq_points) - resample_curve(inactive_points), axis=1)
        row = {'run': run_name, 'stage': stage + 1, 'mean_delta_mm': float(1000.0 * aligned_delta.mean()), 'max_delta_mm': float(1000.0 * aligned_delta.max()), 'hausdorff_mm': float(1000.0 * symmetric_hausdorff(eq_points, inactive_points))}
        stage_rows.append(row)
        print(row)
    endpoint_deltas = 1000.0 * np.linalg.norm(np.asarray(eq_plan['stage_endpoints_world']) - np.asarray(inactive_plan['stage_endpoints_world']), axis=1)
    print({'run': run_name, 'endpoint_deltas_mm': endpoint_deltas.tolist()})

{'run': 'Bssf3', 'stage': 1, 'mean_delta_mm': 0.4281808783346916, 'max_delta_mm': 0.6530651541377944, 'hausdorff_mm': 0.6536774890560318}
{'run': 'Bssf3', 'stage': 2, 'mean_delta_mm': 0.017131137289180223, 'max_delta_mm': 0.05012361653169983, 'hausdorff_mm': 0.05012361653169983}
{'run': 'Bssf3', 'stage': 3, 'mean_delta_mm': 0.2899954019408407, 'max_delta_mm': 0.46453338202551114, 'hausdorff_mm': 0.46453338202551114}
{'run': 'Bssf3', 'stage': 4, 'mean_delta_mm': 1.5517954220144878, 'max_delta_mm': 2.522182865972113, 'hausdorff_mm': 2.19074551712238}
{'run': 'Bssf3', 'stage': 5, 'mean_delta_mm': 11.818952569368433, 'max_delta_mm': 20.980581647389442, 'hausdorff_mm': 18.773633572204787}
{'run': 'Bssf3', 'endpoint_deltas_mm': [0.05012361653169983, 0.001898142490908139, 0.48392699110144655, 0.0010924656081157114, 0.0]}
{'run': 'Bsss1', 'stage': 1, 'mean_delta_mm': 0.05555737077091147, 'max_delta_mm': 0.08190935717740772, 'hausdorff_mm': 0.08192191937717169}
{'run': 'Bsss1', 'stage': 2, 'mea

In [12]:
s4_stage_rows = [row for row in stage_rows if row['stage'] == 4]
s5_stage_rows = [row for row in stage_rows if row['stage'] == 5]
obs_pull_mm = [1000.0 * (row['s4_obs_inactive_mean_m'] - row['s4_obs_eq_mean_m']) for row in ab_rows]
summary_ab = {
    's4_mean_delta_mm_across_runs': float(np.mean([row['mean_delta_mm'] for row in s4_stage_rows])),
    's4_largest_point_delta_mm': float(np.max([row['max_delta_mm'] for row in s4_stage_rows])),
    's4_largest_hausdorff_mm': float(np.max([row['hausdorff_mm'] for row in s4_stage_rows])),
    'obs_eq_pull_toward_target_mean_mm': float(np.mean(obs_pull_mm)),
    'obs_eq_pull_range_mm': [float(np.min(obs_pull_mm)), float(np.max(obs_pull_mm))],
    's5_mean_delta_range_mm': [float(np.min([row['mean_delta_mm'] for row in s5_stage_rows])), float(np.max([row['mean_delta_mm'] for row in s5_stage_rows]))],
    's5_largest_hausdorff_mm': float(np.max([row['hausdorff_mm'] for row in s5_stage_rows])),
}
print(summary_ab)

{'s4_mean_delta_mm_across_runs': 1.5626781399839014, 's4_largest_point_delta_mm': 2.532282140530071, 's4_largest_hausdorff_mm': 2.195870295665215, 'obs_eq_pull_toward_target_mean_mm': 1.3677582408724573, 'obs_eq_pull_range_mm': [1.3393085276660055, 1.4500120520455906], 's5_mean_delta_range_mm': [11.371916294561963, 62.27639898936069], 's5_largest_hausdorff_mm': 97.56918650726277}
